## Additional Resources

- **Documentation**: See `docs/tiling_datamodule_wrapper.md` for comprehensive documentation
- **Examples**: This notebook is the primary runnable example for the tiling workflow
- **PyTorch Docs**: https://pytorch.org/docs/stable/generated/torch.nn.functional.pad.html
- **Memory Layout**: https://pytorch.org/docs/stable/generated/torch.Tensor.is_contiguous.html

## Section 6: Summary of Fixes

### Key Changes Made to `tiled_datamodule_wrapper.py`

The fix addresses the "view size is not compatible" RuntimeError by ensuring tensor contiguity in the custom collate function:

```python
# Fixed padding operations with .contiguous():
padded_v = torch.nn.functional.pad(v, (0, pad_w, 0, pad_h), value=0).contiguous()
```

This change was applied to all three cases in the collate function:
- 2D tensors: [H, W]
- 3D tensors: [C, H, W]
- 4D tensors: [B, C, H, W]

### Why This Fixes the Tests

The tests `test_finetune_multiple_backbones[fit-terramind_v1_base]` and `test_finetune_multiple_backbones_with_prediction[terramind_v1_small]` were failing because:

1. Variable-sized tiles were being padded to match batch dimensions
2. Padded tensors had non-contiguous memory layout
3. Models with reshape operations (.view/.reshape) required contiguous tensors
4. The fix ensures all padded tensors are made contiguous before being stacked and passed to models

### Benefits

- ✓ Fixes RuntimeError in finetune tests
- ✓ Maintains backward compatibility
- ✓ Minimal performance impact (contiguous() is fast when already contiguous)
- ✓ Handles all tensor dimensions correctly
- ✓ Enables proper vision model finetuning with tiled data

In [ ]:
try:
    from terratorch.datasets.tiled_dataset_wrapper import TiledDataset
    
    # Create a tiled dataset
    with tempfile.TemporaryDirectory() as cache_dir:
        print(f"Creating TiledDataset with cache at: {cache_dir}\n")
        
        tiled_dataset = TiledDataset(
            base_dataset=base_dm.train_dataset,
            cache_dir=cache_dir,
            tile_size=(512, 512),
            overlap=64,
            keep_incomplete_tiles=True,
        )
        
        print(f"Original dataset: {len(base_dm.train_dataset)} images")
        print(f"Tiled dataset: {len(tiled_dataset)} tiles")
        print()
        
        # Get a sample tile
        sample = tiled_dataset[0]
        print(f"Sample tile keys: {sample.keys()}")
        print(f"Image shape: {sample['image'].shape}")
        print(f"Mask shape: {sample['mask'].shape}")
        print(f"Tile coordinates (y1, x1, y2, x2): {sample['tile_coords']}")
        print(f"Source image index: {sample['base_idx']}")
        print()
        
        # Verify contiguity
        print(f"Image tensor is_contiguous(): {sample['image'].is_contiguous()}")
        
except ImportError:
    print("⚠ TiledDataset not available in this environment")


## Section 5: Demonstrating TiledDataset

Let's demonstrate how the TiledDataset wrapper processes large images:
- Tiles 1024x1024 images into 512x512 tiles with overlap
- Caches the tiles for fast repeated access
- Generates tile metadata for reconstruction

In [ ]:
class SimpleDataset(Dataset):
    """Mock dataset with large images that benefit from tiling."""
    
    def __init__(self, num_samples=4, image_size=(1024, 1024)):
        self.num_samples = num_samples
        self.image_size = image_size
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        h, w = self.image_size
        # Create a gradient image for visualization
        image = torch.rand(3, h, w)
        mask = torch.randint(0, 5, (h, w))
        return {
            "image": image,
            "mask": mask,
            "filename": f"image_{idx}.tif",
        }


class SimpleDataModule(LightningDataModule):
    """Simple datamodule for testing."""
    
    def __init__(self, batch_size=2, image_size=(1024, 1024)):
        super().__init__()
        self.batch_size = batch_size
        self.image_size = image_size
        self.train_dataset = None
    
    def setup(self, stage=None):
        if stage in ("fit", None):
            self.train_dataset = SimpleDataset(num_samples=4, image_size=self.image_size)
    
    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=None
        )


# Create a simple datamodule
print("Creating simple datamodule...")
base_dm = SimpleDataModule(batch_size=2)
base_dm.setup(stage="fit")
print(f"✓ Created datamodule with {len(base_dm.train_dataset)} samples")
print(f"✓ Each image size: {base_dm.train_dataset.image_size}")


## Section 4: Using TilingDataModuleWrapper

Now let's demonstrate how to use the TilingDataModuleWrapper in practice. This wrapper:
- Automatically tiles large images into manageable chunks
- Caches tiles for efficient repeated access
- Handles variable-sized tiles with the fixed collate function
- Supports prediction stitching for inference

### Mock Dataset for Demonstration

In [ ]:
class SimpleModel(nn.Module):
    """Simple model that performs view operations (tests contiguity requirement)."""
    
    def __init__(self, in_channels=3, num_patches=16):
        super().__init__()
        self.num_patches = num_patches
        self.conv = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.fc = nn.Linear(64 * 256 * 256, 10)
    
    def forward(self, x):
        # x: [B, C, H, W]
        # This view operation requires x to be contiguous
        batch_size = x.shape[0]
        
        x = self.conv(x)  # [B, 64, 256, 256]
        # View operation - this will fail if x is non-contiguous
        x = x.view(batch_size, -1)  # [B, 64*256*256]
        x = self.fc(x)  # [B, 10]
        return x


# Create model and test
model = SimpleModel()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"✓ Model created and moved to {device}")
print(f"✓ Model expects contiguous tensors for view operations")
print()

# Test with padded batch (now fixed with contiguous)
print("Testing model with fixed (contiguous) padded tensors:")
with torch.no_grad():
    collated_batch = collated['image'].to(device)
    try:
        output = model(collated_batch)
        print(f"✓ Model forward pass succeeded! Output shape: {output.shape}")
    except RuntimeError as e:
        print(f"✗ Model forward pass failed: {e}")


## Section 3: Building a Simple Model for Testing

Let's create a simple model that performs reshape operations (like many vision transformers and other models) to test our fixes.

In [ ]:
## Simulating the Fixed Collate Function

def fixed_variable_tile_collate_fn():
    """Create a collate function that handles variable-sized tiles with contiguous tensors."""
    def collate_fn(batch):
        """Collate function that pads variable-sized tiles to max dimensions."""
        if len(batch) == 0:
            return {}
        
        keys = batch[0].keys()
        collated = {}
        
        for key in keys:
            values = [sample[key] for sample in batch]
            
            if isinstance(values[0], torch.Tensor):
                if key in ("image", "mask"):
                    ndims = values[0].ndim
                    
                    if ndims == 3:  # [C, H, W]
                        max_h = max(v.shape[1] for v in values)
                        max_w = max(v.shape[2] for v in values)
                        
                        padded = []
                        for v in values:
                            pad_h = max_h - v.shape[1]
                            pad_w = max_w - v.shape[2]
                            # KEY FIX: Add .contiguous() after padding
                            padded_v = torch.nn.functional.pad(
                                v, (0, pad_w, 0, pad_h), value=0
                            ).contiguous()
                            padded.append(padded_v)
                        collated[key] = torch.stack(padded)
                    else:
                        collated[key] = torch.stack(values)
                else:
                    try:
                        collated[key] = torch.stack(values)
                    except RuntimeError:
                        collated[key] = values
            elif isinstance(values[0], (tuple, list)):
                collated[key] = values
            else:
                collated[key] = torch.tensor(values) if isinstance(values[0], (int, float)) else values
        
        return collated
    
    return collate_fn


# Test the fixed collate function with variable-sized tensors
print("Testing fixed collate function with variable-sized tensors:\n")

# Create mock batch with different sized tiles
batch = [
    {"image": torch.randn(3, 256, 256), "tile_id": 0},  # Full size
    {"image": torch.randn(3, 224, 224), "tile_id": 1},  # Smaller
]

collate_fn = fixed_variable_tile_collate_fn()
collated = collate_fn(batch)

print(f"Input tile sizes:")
print(f"  Tile 0: {batch[0]['image'].shape}")
print(f"  Tile 1: {batch[1]['image'].shape}")
print()
print(f"Collated batch:")
print(f"  Shape: {collated['image'].shape}")
print(f"  is_contiguous(): {collated['image'].is_contiguous()}")
print()

# Verify that models can now reshape the tensor
try:
    reshaped = collated['image'].reshape(-1, 3, 256, 256)[:1]  # Reshape to expected input
    print(f"✓ Model reshape succeeded! Shape: {reshaped.shape}")
except RuntimeError as e:
    print(f"✗ Model reshape failed: {e}")


## Section 2: The Fix Applied to TiledDataModuleWrapper

The custom collate function in TiledDataModuleWrapper handles variable-sized tiles by padding them to match batch dimensions. The fix involves calling `.contiguous()` after each padding operation:

```python
# Before (causes RuntimeError):
padded_v = torch.nn.functional.pad(v, (0, pad_w, 0, pad_h), value=0)

# After (fixed):
padded_v = torch.nn.functional.pad(v, (0, pad_w, 0, pad_h), value=0).contiguous()
```

This ensures that when models try to reshape or view the tensors, they have contiguous memory layout.

In [ ]:
## Demonstrating the Tensor Contiguity Issue

# Create a sample tensor (e.g., padded image tile)
original_tensor = torch.randn(3, 256, 256)  # [C, H, W]
print(f"Original tensor shape: {original_tensor.shape}")
print(f"Original tensor is_contiguous(): {original_tensor.is_contiguous()}")
print()

# Simulate padding (which can create non-contiguous tensors)
padded_tensor = torch.nn.functional.pad(original_tensor, (0, 32, 0, 32), value=0)
print(f"Padded tensor shape: {padded_tensor.shape}")
print(f"Padded tensor is_contiguous(): {padded_tensor.is_contiguous()}")
print()

# Attempting to view a non-contiguous tensor can cause issues
# Let's show what happens
try:
    # Some model operations might try to reshape the tensor
    viewed = padded_tensor.view(-1)  # Flatten
    print("✓ View succeeded (this may work in some cases)")
except RuntimeError as e:
    print(f"✗ View failed with error: {str(e)[:80]}...")
print()

# The fix: use .contiguous()
contiguous_tensor = torch.nn.functional.pad(original_tensor, (0, 32, 0, 32), value=0).contiguous()
print(f"Contiguous tensor is_contiguous(): {contiguous_tensor.is_contiguous()}")
try:
    viewed = contiguous_tensor.view(-1)
    print(f"✓ View succeeded! Flattened shape: {viewed.shape}")
except RuntimeError as e:
    print(f"✗ View failed: {e}")


## Section 1: Understanding the RuntimeError Issue

### The Problem
When using `torch.nn.functional.pad()` on tensors, the resulting tensors may have a non-contiguous memory layout. This occurs because padding can rearrange how data is stored in memory.

When models try to call `.view()` or `.reshape()` on these padded tensors, PyTorch throws:
```
RuntimeError: view size is not compatible with input tensor's size and stride 
(at least one dimension spans across two contiguous subspaces)
```

### The Solution
Call `.contiguous()` after padding to ensure the tensor has contiguous memory layout before passing it to models that expect contiguous tensors.

In [ ]:
import os
import tempfile
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from lightning import LightningDataModule
import logging

# Configure logging to see informative messages
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ PyTorch version:", torch.__version__)
print("✓ CUDA available:", torch.cuda.is_available())

# TilingDataModuleWrapper Tutorial

This notebook demonstrates how to use the TilingDataModuleWrapper to handle large images with consistent padding behavior across training and inference. It also shows how to fix tensor contiguity issues that can cause RuntimeErrors during finetuning.

## Key Topics
1. Understanding tensor contiguity issues in PyTorch
2. Fixing 'view size is not compatible' errors
3. Using TilingDataModuleWrapper for data tiling
4. Running finetune tests with multiple backbones
5. Inference with prediction stitching